# Análise Exploratória dos Dados

## Dicionário de dados

In [1]:
# importar os pacotes necessários
import pandas as pd
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import warnings
import os
warnings.filterwarnings("ignore")

import numpy as np

In [2]:
dicionario = pd.read_csv("../data/raw/dicionario_dados.csv", sep=";")
dicionario

,base,coluna,descricao
0,base_cadastral,id_cliente,Identificador único do cliente
1,base_cadastral,sexo,Sexo do cliente (M ou F)
2,base_cadastral,data_nascimento,Data de nascimento do cliente
3,base_cadastral,qtd_filhos,Quantidade de filhos declarados
4,base_cadastral,qtd_membros_familia,Quantidade total de membros da família
...,...,...,...
64,historico_parcelas,numero_parcela,Número sequencial da parcela dentro do contrato
65,historico_parcelas,data_prevista_pagamento,Data inicialmente prevista para o pagamento da...
66,historico_parcelas,data_real_pagamento,Data real em que o pagamento foi efetuado
67,historico_parcelas,valor_previsto_parcela,Valor original previsto da parcela


## Funções

In [3]:
def agg_hist_contract(hist_contratos, hist_parcelas):
    # unir historico de contratos com parcelas
    hist_total = hist_contratos.merge(
        hist_parcelas, 
        how="left", 
        on=["id_contrato", "id_cliente"]
    )

    hist_total["dias_atraso_parcela"] = (pd.to_datetime(hist_total["data_real_pagamento"]) - pd.to_datetime(hist_total["data_prevista_pagamento"])).dt.days

    hist_total = hist_total.fillna(-1)

    # CRIACAO DA VARIÁVEL ALVO DE ACORDO COM O REGULADOR (BACEN)
    hist_total["target"] = np.where(
        hist_total["dias_atraso_parcela"] > 90,
        1,
        0
    )

    features_temp = hist_total.groupby(
        [
            "id_cliente",
            "id_contrato", 
            "tipo_contrato", 
            "status_contrato", 
            "data_decisao", 
            "valor_credito", 
            "valor_bem",
            "valor_parcela",
            # "percentual_entrada",
            # "qtd_parcelas_planejadas",
            "tipo_pagamento",
            "finalidade_emprestimo",
            "tipo_cliente",
            "faixa_rendimento",
            "tipo_portfolio",
            "tipo_produto",
            "categoria_bem",
            "combinacao_produto",
            "setor_vendedor",
            "dia_semana_solicitacao",
            "hora_solicitacao",
            # "flag_ultima_solicitacao_contrato",
            # "flag_ultima_solicitacao_dia",
            "motivo_recusa",
            # "acompanhantes_cliente",
            # "flag_seguro_contratado"
        ]
    ).agg(
    num_max_parcelas=("numero_parcela", "max"),
    valor_total_a_pagar=("valor_parcela", "sum"),
    valor_medio_pago_parcela=("valor_pago_parcela", "mean"),
    valor_max_pago_parcela=("valor_pago_parcela", "max"),
    target=("target", "max"),
    ).reset_index()

    features_temp = features_temp.sort_values(by=["id_cliente", "data_decisao"])
    return features_temp

## Criação das Métricas Históricas

In [4]:
# importar as tabelas
cadastral = pd.read_parquet('../data/raw/base_cadastral.parquet', engine="fastparquet")
submissao = pd.read_parquet('../data/raw/base_submissao.parquet', engine="fastparquet") 
hist_contratos = pd.read_parquet('../data/raw/historico_emprestimos.parquet', engine="fastparquet")
hist_parcelas = pd.read_parquet('../data/raw/historico_parcelas.parquet', engine="fastparquet")

### 1. Agregar Histórico por Contrato

In [6]:
df_hist_contract = agg_hist_contract(hist_contratos, hist_parcelas)
df_hist_contract

,id_cliente,id_contrato,tipo_contrato,status_contrato,data_decisao,valor_credito,valor_bem,valor_parcela,tipo_pagamento,finalidade_emprestimo,tipo_cliente,faixa_rendimento,tipo_portfolio,tipo_produto,categoria_bem,combinacao_produto,setor_vendedor,dia_semana_solicitacao,hora_solicitacao,motivo_recusa,num_max_parcelas,valor_total_a_pagar,valor_medio_pago_parcela,valor_max_pago_parcela,target
0,100023,1131053,Consumer loans,Approved,2018-07-07,76680.0,78705.0,8519.130,Cash through the bank,XAP,Repeater,low_normal,POS,XNA,Consumer Electronics,POS household without interest,Consumer electronics,SATURDAY,10,XAP,8.0,68153.040,10575.292500,24968.430,0
2,100023,2411919,Consumer loans,Approved,2020-02-01,93145.5,91282.5,7992.000,Cash through the bank,XAP,Repeater,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,SATURDAY,10,XAP,14.0,111888.000,9046.992857,22761.900,0
1,100023,1499902,Revolving loans,Approved,2024-04-03,45000.0,45000.0,2250.000,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,WEDNESDAY,11,XAP,-1.0,2250.000,-1.000000,-1.000,0
3,100023,2454202,Cash loans,Approved,2024-06-07,239242.5,180000.0,16822.440,Cash through the bank,XNA,Repeater,high,Cash,x-sell,XNA,Cash X-Sell: high,XNA,FRIDAY,9,XAP,4.0,67289.760,70571.711250,231819.525,0
4,100056,1636219,Consumer loans,Approved,2017-05-25,201528.0,223920.0,13502.385,Cash through the bank,XAP,New,high,POS,XNA,Computers,POS household with interest,Consumer electronics,THURSDAY,8,XAP,101.0,189033.390,22716.144643,168149.790,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,456245,1666363,Cash loans,Approved,2024-07-12,116077.5,90000.0,6481.755,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,FRIDAY,10,XAP,7.0,45372.285,6481.755000,6481.755,0
186887,456248,1395578,Consumer loans,Approved,2017-10-05,41940.0,39960.0,8417.340,Cash through the bank,XAP,New,high,POS,XNA,Mobile,POS mobile with interest,Connectivity,THURSDAY,14,XAP,6.0,84173.400,5047.740000,8417.340,0
186888,456248,1826280,Cash loans,Approved,2022-01-10,271300.5,225000.0,28633.320,Cash through the bank,XNA,Refreshed,middle,Cash,x-sell,XNA,Cash X-Sell: middle,XNA,MONDAY,13,XAP,12.0,343599.840,28608.903750,28633.320,0
186886,456248,1136073,Cash loans,Approved,2023-01-27,1333179.0,1260000.0,67725.495,Cash through the bank,XNA,Repeater,low_action,Cash,x-sell,XNA,Cash X-Sell: low,XNA,FRIDAY,16,XAP,24.0,1625411.880,67709.353125,67725.495,0


In [7]:
df_hist_contract.target.value_counts()

target
0    186054
1       836
Name: count, dtype: int64

### 2. Criar Janela Temporal Agregando as Métricas por Janela de "data_decisao"

In [8]:
# CONFIG
COLUNAS_ULTIMO_VALOR = [
    # "tipo_contrato",
    "status_contrato",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    "motivo_recusa",
    # "acompanhantes_cliente",
    # "flag_seguro_contratado",
    # "flag_ultima_solicitacao_contrato",
    # "flag_ultima_solicitacao_dia",
    "dia_semana_solicitacao",
    "hora_solicitacao",
]

AGGS = {
    "valor_credito": ["mean", "max", "min"],
    "valor_bem": ["mean", "max", "min"],
    "valor_parcela": ["mean", "max", "min"],
    # "percentual_entrada": ["mean"],
    # "qtd_parcelas_planejadas": ["max"],
    "num_max_parcelas": ["mean", "max"],
    "valor_total_a_pagar": ["mean", "sum"],
    "valor_medio_pago_parcela": ["mean"],
    "valor_max_pago_parcela": ["max"],
}

COLUNAS_REMOVER = [
    "status_contrato",
    # "percentual_entrada",
    # "qtd_parcelas_planejadas",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    # "flag_ultima_solicitacao_contrato",
    # "flag_ultima_solicitacao_dia",
    "motivo_recusa",
    # "acompanhantes_cliente",
    # "flag_seguro_contratado",
    "num_max_parcelas",
    "valor_total_a_pagar",
    "valor_medio_pago_parcela",
    "valor_max_pago_parcela",
    "hist_ultimo_dia_semana_solicitacao",
    "hist_ultimo_hora_solicitacao"
]


def create_rolling_features(df: pd.DataFrame, colunas_remover: list) -> pd.DataFrame:
    
    df = df.sort_values(["id_cliente", "data_decisao"]).copy()
    group = df.groupby("id_cliente")

    # QUANTIDADE DE CONTRATOS
    df["hist_qtd_contratos"] = group.cumcount().replace(0, np.nan)

    # FEATURES NUMÉRICAS (EXPANDING)
    for col, funcs in AGGS.items():
        tmp = (
            group[col]
            .expanding()
            .agg(funcs)
            .shift(1)
            .reset_index(level=0, drop=True)
        )

        # Caso venha como Series (apenas 1 agg)
        if isinstance(tmp, pd.Series):
            df[f"hist_{col}_{funcs[0]}"] = tmp
        else:
            for f in funcs:
                df[f"hist_{col}_{f}"] = tmp[f]

    # ÚLTIMO VALOR (SHIFT)
    for col in COLUNAS_ULTIMO_VALOR:
        df[f"hist_ultimo_{col}"] = group[col].shift(1)


    # GARANTIA: PRIMEIRA LINHA = NaN
    first_idx = group.head(1).index
    cols_hist = [c for c in df.columns if c.startswith("hist_")]

    df.loc[first_idx, cols_hist] = np.nan
    df = df.drop(columns=colunas_remover, errors="ignore")

    return df

df_window = create_rolling_features(df_hist_contract, COLUNAS_REMOVER)

### 3. Unir dados de janela com os dados cadastrais

In [10]:
df_window_final = df_window.merge(
    cadastral, 
    how="left", 
    on="id_cliente"
)

df_window_final = df_window_final.drop(["sexo", "data_nascimento"], axis=1)

### 4. Criar tabelas de treino e teste

In [11]:
df_window_final = df_window_final.sort_values(
    ["id_cliente", "data_decisao", "hist_qtd_contratos", "id_contrato"]
)

In [12]:
df_test = df_window_final.groupby("id_cliente").tail(1).copy()

df_test = df_test.drop([
    "tipo_contrato",
    "valor_credito",
    "valor_bem",
    "valor_parcela",
    "dia_semana_solicitacao",
    "hora_solicitacao",
    "target",
    "id_contrato",
    "data_decisao"
], axis=1)

df_test = submissao.merge(df_test, how="left", on="id_cliente")

df_test = df_test.drop([
    "data_solicitacao"
], axis=1)

df_test = df_test.replace(-1, np.nan)

In [14]:
df_train = df_window_final.drop(df_test.index)
df_train = df_train.drop([
    "id_cliente",
    "id_contrato",
    "data_decisao"
], axis=1)

df_train = df_train.replace(-1, np.nan)

In [16]:
output_dir = "../data/curated"
os.makedirs(output_dir, exist_ok=True)
df_train.to_parquet(f"{output_dir}/train.parquet", index=False)
df_test.to_parquet(f"{output_dir}/test.parquet", index=False)